# Week 13: Logistic Regression for Classification

**ISM4641 - Python for Business Analytics**

---

## Introduction

In our journey through data analytics, we've learned how to predict continuous values using linear regression—forecasting sales, estimating prices, projecting revenues. But many of the most critical business decisions involve a different type of question: **Will this customer churn?** **Will this loan default?** **Will this transaction be fraudulent?** These are yes/no questions that require a fundamentally different approach.

Welcome to the world of **classification**—one of the most widely applied areas of machine learning in business. In this chapter, we explore **logistic regression**, a powerful yet interpretable algorithm that predicts categorical outcomes while providing probability estimates. Despite its name, logistic regression is a classification algorithm, not a regression algorithm.

Logistic regression sits at a fascinating intersection of statistics and machine learning. It's mathematically elegant, computationally efficient, and—perhaps most importantly for business applications—highly interpretable. When you build a logistic regression model to predict customer churn, you can explain to stakeholders exactly which factors increase or decrease the likelihood of churn and by how much.

By mastering logistic regression, you'll be equipped to tackle some of the most common and valuable problems in business analytics: identifying at-risk customers, detecting fraud, qualifying leads, predicting loan defaults, and optimizing marketing campaigns. Let's begin our exploration of classification.

---

## Learning Objectives

By the end of this chapter, you will be able to:

1. **Distinguish** between regression and classification problems in business contexts
2. **Understand** the mathematical foundation of logistic regression (sigmoid function, log-odds)
3. **Build** logistic regression models using scikit-learn for binary classification
4. **Interpret** model coefficients as odds ratios with business meaning
5. **Evaluate** classification models using accuracy, precision, recall, F1-score, and confusion matrices
6. **Apply** logistic regression to real-world problems: churn prediction, marketing response, and loan default
7. **Communicate** model results effectively to non-technical stakeholders

## Table of Contents

- [Week 13: Logistic Regression for Classification](#week-13-logistic-regression-for-classification)
  - [Introduction](#introduction)
  - [Learning Objectives](#learning-objectives)
  - [13.1 Classification vs. Regression: Understanding the Fundamental Difference](#131-classification-vs-regression-understanding-the-fundamental-difference)
  - [13.2 The Mathematics of Logistic Regression](#132-the-mathematics-of-logistic-regression)
  - [13.3 Building Your First Logistic Regression Model](#133-building-your-first-logistic-regression-model)
  - [13.4 Evaluating Classification Models](#134-evaluating-classification-models)
  - [13.5 Interpreting Coefficients: Odds Ratios](#135-interpreting-coefficients-odds-ratios)
  - [13.6 Practical Application: Marketing Campaign Response](#136-practical-application-marketing-campaign-response)
  - [13.7 Practical Application: Loan Default Prediction](#137-practical-application-loan-default-prediction)
  - [13.8 Common Pitfalls and Best Practices](#138-common-pitfalls-and-best-practices)
  - [13.9 Ethical Implications of Classification Models](#139-ethical-implications-of-classification-models)
  - [13.10 Complete Workflow: A Business Case Study](#1310-complete-workflow-a-business-case-study)
  - [13.11 Summary](#1311-summary)
  - [Practice Exercises](#practice-exercises)
  - [Course Conclusion](#course-conclusion)

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)

# Data repository URL for course datasets
DATA_URL = "https://raw.githubusercontent.com/prof-tcsmith/ism6251s26-data/main"

print("Libraries imported successfully!")
print(f"Data URL: {DATA_URL}")

---

## 13.1 Classification vs. Regression: Understanding the Fundamental Difference

Before diving into logistic regression, we need to clearly understand the distinction between **regression** and **classification** problems. This distinction determines which algorithms we can use and how we evaluate our models.

### Regression: Predicting Continuous Values

In regression problems, we predict a **continuous numerical value** that can take any value within a range:

- **House price**: $150,000, $275,500, $1,234,567.89
- **Sales revenue**: $45,000, $127,892.50
- **Temperature**: 72.5°F, 68.3°F
- **Customer lifetime value**: $500, $12,750

Linear regression, which we studied last week, is designed for these problems.

### Classification: Predicting Categories

In classification problems, we predict **discrete categories** or **class labels**:

- **Customer status**: Churn / Stay (binary)
- **Email type**: Spam / Not Spam (binary)
- **Loan outcome**: Default / Repaid (binary)
- **Customer segment**: Bronze / Silver / Gold / Platinum (multi-class)
- **Sentiment**: Negative / Neutral / Positive (multi-class)

### Binary vs. Multi-class Classification

| Type | Classes | Examples |
|------|---------|----------|
| **Binary** | 2 | Yes/No, True/False, 0/1 |
| **Multi-class** | 3+ | Low/Medium/High, Product categories |

In this chapter, we focus on **binary classification**—the most common scenario in business applications. Logistic regression naturally handles binary outcomes and can be extended to multi-class problems.

### Why Can't We Use Linear Regression for Classification?

You might wonder: if we code "Churn" as 1 and "Stay" as 0, can't we just use linear regression?

Let's see why this doesn't work:

In [ ]:
# Why linear regression fails for classification
np.random.seed(42)

# Simple example: tenure (months) vs churn (0/1)
tenure = np.array([1, 2, 3, 6, 12, 18, 24, 36, 48, 60])
churn = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])  # Short tenure = churn

# Fit linear regression
from sklearn.linear_model import LinearRegression
lin_model = LinearRegression()
lin_model.fit(tenure.reshape(-1, 1), churn)

# Generate predictions for a range of tenure values
tenure_range = np.linspace(0, 80, 100)
lin_pred = lin_model.predict(tenure_range.reshape(-1, 1))

# Visualize the problem
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(tenure, churn, s=100, c='blue', label='Actual Data')
plt.plot(tenure_range, lin_pred, 'r-', linewidth=2, label='Linear Regression')
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('Tenure (months)')
plt.ylabel('Churn (0=Stay, 1=Churn)')
plt.title('Problem: Linear Regression for Classification')
plt.legend()
plt.ylim(-0.3, 1.3)

# Show problematic predictions
plt.subplot(1, 2, 2)
problematic_tenure = np.array([0, 70, 80])
problematic_pred = lin_model.predict(problematic_tenure.reshape(-1, 1))
colors = ['red' if p < 0 or p > 1 else 'green' for p in problematic_pred]
plt.bar(['0 months', '70 months', '80 months'], problematic_pred, color=colors)
plt.axhline(y=0, color='black', linestyle='--')
plt.axhline(y=1, color='black', linestyle='--')
plt.ylabel('Predicted "Probability"')
plt.title('Invalid Predictions Outside [0, 1]')
for i, (t, p) in enumerate(zip(['0 mo', '70 mo', '80 mo'], problematic_pred)):
    plt.text(i, p + 0.05, f'{p:.2f}', ha='center', fontsize=12)

plt.tight_layout()
plt.show()

print("Problems with Linear Regression for Classification:")
print("1. Predictions can be < 0 (impossible probability)")
print("2. Predictions can be > 1 (impossible probability)")
print("3. The relationship isn't actually linear")

### The Problems with Linear Regression for Classification

1. **Predictions outside [0, 1]**: Probabilities must be between 0 and 1, but linear regression can predict any value.

2. **Wrong functional form**: The relationship between features and probability of an event is rarely linear. The probability of churn doesn't decrease at a constant rate with tenure—it levels off.

3. **Sensitive to outliers**: A few extreme values can dramatically shift the regression line.


We need an approach that:
- Constrains predictions to [0, 1]
- Models the S-shaped relationship often seen in probability
- Provides a natural interpretation as probability

Enter **logistic regression**.

---

## 13.2 The Mathematics of Logistic Regression

Logistic regression solves the problems of linear regression for classification through a clever mathematical transformation.

### The Sigmoid (Logistic) Function

The **sigmoid function** transforms any input value to an output between 0 and 1:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Where:
- $z$ is the linear combination of features: $z = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_n x_n$
- $e$ is Euler's number (approximately 2.718)
- The output $\sigma(z)$ is always between 0 and 1

### Properties of the Sigmoid Function

| Property | Value/Description |
|----------|-------------------|
| When $z = 0$ | $\sigma(0) = 0.5$ |
| As $z \to +\infty$ | $\sigma(z) \to 1$ |
| As $z \to -\infty$ | $\sigma(z) \to 0$ |
| Shape | S-curve (sigmoid) |
| Center | Always passes through (0, 0.5) |

In [ ]:
# Define and visualize the sigmoid function
def sigmoid(z):
    """The logistic (sigmoid) function."""
    return 1 / (1 + np.exp(-z))

# Generate z values
z = np.linspace(-10, 10, 200)
p = sigmoid(z)

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Basic sigmoid function
ax1 = axes[0]
ax1.plot(z, p, 'b-', linewidth=2.5, label='Sigmoid function')
ax1.axhline(y=0.5, color='red', linestyle='--', alpha=0.7, label='Decision boundary (0.5)')
ax1.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax1.axhline(y=1, color='gray', linestyle=':', alpha=0.5)
ax1.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
ax1.fill_between(z, 0, p, where=(p < 0.5), alpha=0.2, color='green', label='Predict 0 (Stay)')
ax1.fill_between(z, 0, p, where=(p >= 0.5), alpha=0.2, color='red', label='Predict 1 (Churn)')
ax1.set_xlabel('z (Linear Combination)', fontsize=12)
ax1.set_ylabel('P(y=1) - Probability', fontsize=12)
ax1.set_title('The Sigmoid Function', fontsize=14)
ax1.legend(loc='upper left')
ax1.set_ylim(-0.05, 1.05)
ax1.grid(True, alpha=0.3)

# Plot 2: Key points on the sigmoid
ax2 = axes[1]
ax2.plot(z, p, 'b-', linewidth=2.5)

# Mark key points
key_z = [-4, -2, 0, 2, 4]
key_p = sigmoid(np.array(key_z))

for zi, pi in zip(key_z, key_p):
    ax2.plot(zi, pi, 'ro', markersize=10)
    ax2.annotate(f'z={zi}: P={pi:.3f}', 
                xy=(zi, pi), xytext=(zi+0.5, pi+0.08),
                fontsize=10, ha='left')

ax2.set_xlabel('z (Linear Combination)', fontsize=12)
ax2.set_ylabel('P(y=1) - Probability', fontsize=12)
ax2.set_title('Key Points on the Sigmoid Curve', fontsize=14)
ax2.set_ylim(-0.05, 1.15)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print key values
print("\nKey Sigmoid Values:")
print("-" * 35)
for zi in [-6, -4, -2, -1, 0, 1, 2, 4, 6]:
    print(f"z = {zi:3d}  →  P(y=1) = {sigmoid(zi):.4f}")

### Understanding Log-Odds (Logit)

The mathematics of logistic regression involves the concept of **odds** and **log-odds** (logit).

#### From Probability to Odds

If the probability of an event is $p$, then:

$$\text{Odds} = \frac{p}{1-p}$$

Examples:
- If P(Churn) = 0.5, Odds = 0.5/0.5 = 1 ("even odds")
- If P(Churn) = 0.8, Odds = 0.8/0.2 = 4 ("4 to 1 odds of churning")
- If P(Churn) = 0.2, Odds = 0.2/0.8 = 0.25 ("1 to 4 odds of churning")

#### From Odds to Log-Odds (Logit)

$$\text{Log-Odds} = \log\left(\frac{p}{1-p}\right) = \text{logit}(p)$$

The log-odds can be any value from $-\infty$ to $+\infty$, which is exactly what our linear combination $z$ produces!

### The Logistic Regression Model

Logistic regression models the **log-odds** as a linear function of features:

$$\log\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_n x_n$$

Solving for $p$, we get:

$$p = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + ... + \beta_n x_n)}}$$

This is the sigmoid function applied to our linear combination!

In [ ]:
# Visualize the relationship: probability ↔ odds ↔ log-odds
prob = np.linspace(0.001, 0.999, 200)  # Avoid 0 and 1 for log
odds = prob / (1 - prob)
log_odds = np.log(odds)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Probability to Odds
axes[0].plot(prob, odds, 'b-', linewidth=2)
axes[0].axhline(y=1, color='red', linestyle='--', alpha=0.7, label='Odds = 1 (even odds)')
axes[0].set_xlabel('Probability', fontsize=12)
axes[0].set_ylabel('Odds', fontsize=12)
axes[0].set_title('Probability → Odds', fontsize=14)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 10)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Probability to Log-Odds
axes[1].plot(prob, log_odds, 'g-', linewidth=2)
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.7, label='Log-odds = 0 (P=0.5)')
axes[1].set_xlabel('Probability', fontsize=12)
axes[1].set_ylabel('Log-Odds (Logit)', fontsize=12)
axes[1].set_title('Probability → Log-Odds', fontsize=14)
axes[1].set_xlim(0, 1)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Table of examples
axes[2].axis('off')
example_probs = [0.1, 0.2, 0.5, 0.8, 0.9]
table_data = []
for p in example_probs:
    o = p / (1 - p)
    lo = np.log(o)
    table_data.append([f'{p:.1f}', f'{o:.2f}', f'{lo:.2f}'])

table = axes[2].table(
    cellText=table_data,
    colLabels=['Probability', 'Odds', 'Log-Odds'],
    loc='center',
    cellLoc='center',
    colWidths=[0.3, 0.3, 0.3]
)
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 1.8)
axes[2].set_title('Example Conversions', fontsize=14, y=0.85)

plt.tight_layout()
plt.show()

---

## 13.3 Building Your First Logistic Regression Model

Now let's build a practical logistic regression model using real business data. We'll predict **customer churn**—a critical problem for any subscription-based business.

### The Customer Churn Problem

Customer churn (also called customer attrition) refers to customers who stop doing business with a company. Understanding and predicting churn is valuable because:

1. **It's expensive to acquire new customers** - typically 5-25x more expensive than retaining existing ones
2. **Early intervention can prevent churn** - if we know who's likely to leave, we can take action
3. **Not all customers are equally valuable** - we should prioritize retention efforts

### Loading and Exploring the Data

In [ ]:
# Load customer churn data from remote repository
churn_data = pd.read_csv(f"{DATA_URL}/W13/customer_churn.csv")

print("Customer Churn Dataset")
print("=" * 50)
print(f"Total customers: {len(churn_data):,}")
print(f"\nFeatures available:")
for col in churn_data.columns:
    print(f"  • {col}: {churn_data[col].dtype}")

print("\n" + "=" * 50)
print("Sample Data:")
display(churn_data.head(10))

In [ ]:
# Explore the target variable: Churn distribution
churn_counts = churn_data['Churned'].value_counts()
churn_pct = churn_data['Churned'].value_counts(normalize=True) * 100

print("Churn Distribution:")
print("-" * 30)
print(f"Stayed (0):  {churn_counts[0]:>4} ({churn_pct[0]:.1f}%)")
print(f"Churned (1): {churn_counts[1]:>4} ({churn_pct[1]:.1f}%)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
colors = ['#2ecc71', '#e74c3c']
axes[0].pie([churn_counts[0], churn_counts[1]], 
           labels=['Stayed', 'Churned'],
           colors=colors,
           autopct='%1.1f%%',
           startangle=90,
           explode=(0, 0.05))
axes[0].set_title('Customer Churn Distribution', fontsize=14)

# Bar chart
axes[1].bar(['Stayed (0)', 'Churned (1)'], 
           [churn_counts[0], churn_counts[1]], 
           color=colors)
axes[1].set_ylabel('Number of Customers', fontsize=12)
axes[1].set_title('Churn Counts', fontsize=14)
for i, (count, pct) in enumerate(zip([churn_counts[0], churn_counts[1]], 
                                      [churn_pct[0], churn_pct[1]])):
    axes[1].text(i, count + 5, f'{count}\n({pct:.1f}%)', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Explore features and their relationship to churn
print("Feature Statistics by Churn Status:")
print("=" * 60)
display(churn_data.groupby('Churned').agg({
    'Tenure': ['mean', 'std'],
    'MonthlyCharges': ['mean', 'std'],
    'SupportCalls': ['mean', 'std']
}).round(2))

# Visualize feature distributions by churn status
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

features_to_plot = ['Tenure', 'MonthlyCharges', 'SupportCalls']
colors = {0: '#2ecc71', 1: '#e74c3c'}

for ax, feature in zip(axes, features_to_plot):
    for churn_val in [0, 1]:
        data = churn_data[churn_data['Churned'] == churn_val][feature]
        label = 'Stayed' if churn_val == 0 else 'Churned'
        ax.hist(data, bins=20, alpha=0.6, color=colors[churn_val], 
               label=label, density=True)
    ax.set_xlabel(feature, fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.set_title(f'{feature} Distribution by Churn', fontsize=14)
    ax.legend()

plt.tight_layout()
plt.show()

print("\nObservations:")
print("• Customers who churned tend to have shorter tenure")
print("• Higher monthly charges may be associated with higher churn")
print("• More support calls tend to indicate higher churn risk")

### Training the Logistic Regression Model

The scikit-learn workflow for logistic regression is very similar to linear regression:

1. **Prepare data**: Separate features (X) and target (y)
2. **Split data**: Training and testing sets
3. **Create model**: Instantiate LogisticRegression
4. **Train model**: Call fit() method
5. **Make predictions**: Call predict() or predict_proba()
6. **Evaluate**: Assess model performance

In [ ]:
# Step 1: Prepare features and target
feature_columns = ['Tenure', 'MonthlyCharges', 'SupportCalls']
X = churn_data[feature_columns]
y = churn_data['Churned']

print("Step 1: Data Preparation")
print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")
print(f"Features used: {feature_columns}")

In [ ]:
# Step 2: Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 20% for testing
    random_state=42,    # For reproducibility
    stratify=y          # Maintain class proportions
)

print("Step 2: Train/Test Split")
print(f"Training samples: {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)")
print(f"Testing samples: {len(X_test)} ({len(X_test)/len(X)*100:.0f}%)")
print(f"\nChurn rate in training: {y_train.mean()*100:.1f}%")
print(f"Churn rate in testing: {y_test.mean()*100:.1f}%")
print("(Stratification preserved class balance)")

In [ ]:
# Step 3 & 4: Create and train the model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

print("Step 3 & 4: Model Training")
print("LogisticRegression model trained successfully!")
print(f"\nModel parameters:")
print(f"  Intercept (β₀): {model.intercept_[0]:.4f}")
print(f"  Coefficients:")
for feature, coef in zip(feature_columns, model.coef_[0]):
    print(f"    {feature}: {coef:.4f}")

In [ ]:
# Step 5: Make predictions
# Class predictions (0 or 1)
y_pred = model.predict(X_test)

# Probability predictions
y_prob = model.predict_proba(X_test)

print("Step 5: Making Predictions")
print("\nPrediction Types:")
print("  • predict(): Returns class labels (0 or 1)")
print("  • predict_proba(): Returns probabilities [P(Stay), P(Churn)]")

# Show first 10 predictions with probabilities
results_df = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': y_pred[:10],
    'P(Stay)': y_prob[:10, 0].round(3),
    'P(Churn)': y_prob[:10, 1].round(3),
    'Correct?': ['✓' if a == p else '✗' for a, p in zip(y_test.values[:10], y_pred[:10])]
})

print("\nFirst 10 Test Predictions:")
display(results_df)

---

## 13.4 Evaluating Classification Models

Classification model evaluation is more nuanced than regression. While we have a single number (accuracy), it often doesn't tell the whole story. Let's explore the full toolkit of classification metrics.

### Accuracy: The Basics

**Accuracy** is the proportion of correct predictions:

$$\text{Accuracy} = \frac{\text{Number of Correct Predictions}}{\text{Total Predictions}}$$

In [ ]:
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy")
print("=" * 40)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")
print(f"\nInterpretation:")
print(f"  The model correctly predicts churn status")
print(f"  for {accuracy*100:.1f}% of customers.")

### The Accuracy Paradox

Accuracy can be misleading, especially with **imbalanced classes**. Consider this scenario:

- If only 5% of customers churn, a model that predicts "no churn" for everyone achieves 95% accuracy!
- But this model is useless—it never identifies at-risk customers.

We need more detailed metrics to understand model performance.

### The Confusion Matrix: A Complete Picture

The **confusion matrix** shows the breakdown of predictions:

|  | **Predicted: Stay (0)** | **Predicted: Churn (1)** |
|--|-------------------------|-------------------------|
| **Actual: Stay (0)** | True Negative (TN) | False Positive (FP) |
| **Actual: Churn (1)** | False Negative (FN) | True Positive (TP) |

**Definitions:**
- **True Positive (TP)**: Correctly predicted churn
- **True Negative (TN)**: Correctly predicted stay
- **False Positive (FP)**: Predicted churn, but actually stayed (Type I error)
- **False Negative (FN)**: Predicted stay, but actually churned (Type II error)

In [ ]:
# Create and visualize confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Extract values
TN, FP, FN, TP = cm.ravel()

print("Confusion Matrix")
print("=" * 50)
print(f"\n{'':<20} Predicted Stay  Predicted Churn")
print(f"{'Actual Stay':<20} {TN:^14} {FP:^14}")
print(f"{'Actual Churn':<20} {FN:^14} {TP:^14}")
print(f"\nTrue Negatives (TN): {TN} - Correctly identified as staying")
print(f"True Positives (TP): {TP} - Correctly identified as churning")
print(f"False Positives (FP): {FP} - Wrongly flagged as churning (false alarm)")
print(f"False Negatives (FN): {FN} - Missed churners (the dangerous ones!)")

# Visualize
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)

classes = ['Stay (0)', 'Churn (1)']
ax.set(xticks=[0, 1], yticks=[0, 1],
       xticklabels=classes, yticklabels=classes,
       xlabel='Predicted Label', ylabel='True Label',
       title='Confusion Matrix')

# Add text annotations
labels = [['TN', 'FP'], ['FN', 'TP']]
for i in range(2):
    for j in range(2):
        text_color = 'white' if cm[i, j] > cm.max()/2 else 'black'
        ax.text(j, i, f'{labels[i][j]}\n{cm[i, j]}',
               ha='center', va='center', color=text_color, fontsize=14)

plt.tight_layout()
plt.show()

### Precision, Recall, and F1-Score

These metrics provide different perspectives on classification performance:

| Metric | Formula | Question It Answers |
|--------|---------|--------------------|
| **Precision** | TP / (TP + FP) | Of all customers we predicted would churn, how many actually did? |
| **Recall** | TP / (TP + FN) | Of all customers who actually churned, how many did we identify? |
| **F1-Score** | 2 × (P × R) / (P + R) | Harmonic mean of precision and recall |

### Business Interpretation

**High Precision is important when:**
- False positives are costly
- Retention offers are expensive
- You want to minimize wasted resources on non-churners

**High Recall is important when:**
- Missing a churner is very costly
- Customer lifetime value is high
- You'd rather offer retention to some non-churners than miss actual churners

In [ ]:
# Calculate precision, recall, and F1
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Classification Metrics")
print("=" * 50)
print(f"\nPrecision: {precision:.4f} ({precision*100:.1f}%)")
print(f"  → Of {TP + FP} customers predicted to churn,")
print(f"    {TP} actually did ({precision*100:.1f}% were correct).")

print(f"\nRecall: {recall:.4f} ({recall*100:.1f}%)")
print(f"  → Of {TP + FN} customers who actually churned,")
print(f"    we identified {TP} ({recall*100:.1f}%).")

print(f"\nF1-Score: {f1:.4f}")
print(f"  → Balanced measure combining precision and recall.")

# Visualize the metrics
fig, ax = plt.subplots(figsize=(10, 5))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
values = [accuracy, precision, recall, f1]
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

bars = ax.bar(metrics, values, color=colors)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Classification Performance Metrics', fontsize=14)
ax.set_ylim(0, 1.1)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)

# Add value labels
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
           f'{val:.3f}', ha='center', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Complete classification report
print("Complete Classification Report")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=['Stay', 'Churn']))

### Understanding the Classification Report

The classification report shows metrics for each class:

- **Support**: Number of actual instances of each class in the test set
- **Macro avg**: Simple average of class metrics (treats classes equally)
- **Weighted avg**: Average weighted by support (accounts for class imbalance)

---

## 13.5 Interpreting Coefficients: Odds Ratios

One of the greatest strengths of logistic regression is its **interpretability**. Unlike black-box models, we can explain exactly how each feature influences the prediction.

### From Coefficients to Odds Ratios

The coefficients in logistic regression represent **log-odds**. To make them interpretable, we convert them to **odds ratios**:

$$\text{Odds Ratio} = e^{\text{coefficient}}$$

### Interpreting Odds Ratios

| Odds Ratio | Interpretation |
|------------|---------------|
| **OR = 1** | No effect on odds |
| **OR > 1** | Increases the odds (e.g., OR=2 means doubles the odds) |
| **OR < 1** | Decreases the odds (e.g., OR=0.5 means halves the odds) |

In [ ]:
# Calculate and display odds ratios
print("Coefficient Interpretation")
print("=" * 70)

# Create DataFrame with coefficients and odds ratios
coef_df = pd.DataFrame({
    'Feature': feature_columns,
    'Coefficient': model.coef_[0],
    'Odds Ratio': np.exp(model.coef_[0]),
    'Effect': ['Increases' if c > 0 else 'Decreases' for c in model.coef_[0]]
})

print(f"\nIntercept (β₀): {model.intercept_[0]:.4f}")
print("\nFeature Coefficients and Odds Ratios:")
display(coef_df)

# Detailed interpretations
print("\n" + "=" * 70)
print("Business Interpretations:")
print("-" * 70)

for _, row in coef_df.iterrows():
    feature = row['Feature']
    coef = row['Coefficient']
    odds_ratio = row['Odds Ratio']
    
    if odds_ratio > 1:
        pct_change = (odds_ratio - 1) * 100
        direction = "increases"
    else:
        pct_change = (1 - odds_ratio) * 100
        direction = "decreases"
    
    print(f"\n{feature}:")
    print(f"  Coefficient: {coef:.4f}")
    print(f"  Odds Ratio: {odds_ratio:.4f}")
    
    if feature == 'Tenure':
        print(f"  → Each additional month of tenure {direction} the odds")
        print(f"    of churn by {pct_change:.1f}%.")
        print(f"  → Long-term customers are more loyal.")
    elif feature == 'MonthlyCharges':
        print(f"  → Each $1 increase in monthly charges {direction} the odds")
        print(f"    of churn by {pct_change:.2f}%.")
        print(f"  → Higher-paying customers may be more price-sensitive.")
    elif feature == 'SupportCalls':
        print(f"  → Each additional support call {direction} the odds")
        print(f"    of churn by {pct_change:.1f}%.")
        print(f"  → Frequent support contact indicates frustration.")

In [ ]:
# Visualize feature importance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Coefficients (log-odds)
ax1 = axes[0]
colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in coef_df['Coefficient']]
bars = ax1.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
ax1.axvline(x=0, color='black', linewidth=1)
ax1.set_xlabel('Coefficient (Log-Odds)', fontsize=12)
ax1.set_title('Feature Coefficients\n(Red = Increases Churn, Green = Decreases Churn)', fontsize=13)
ax1.grid(True, alpha=0.3, axis='x')

# Add value labels
for bar, coef in zip(bars, coef_df['Coefficient']):
    width = bar.get_width()
    ax1.text(width + 0.01 if width >= 0 else width - 0.01, bar.get_y() + bar.get_height()/2,
            f'{coef:.4f}', ha='left' if width >= 0 else 'right', va='center', fontsize=11)

# Plot 2: Odds Ratios
ax2 = axes[1]
colors = ['#e74c3c' if or_ > 1 else '#2ecc71' for or_ in coef_df['Odds Ratio']]
bars = ax2.barh(coef_df['Feature'], coef_df['Odds Ratio'], color=colors)
ax2.axvline(x=1, color='black', linewidth=1, linestyle='--', label='OR = 1 (no effect)')
ax2.set_xlabel('Odds Ratio', fontsize=12)
ax2.set_title('Feature Odds Ratios\n(>1 Increases Churn, <1 Decreases Churn)', fontsize=13)
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3, axis='x')

# Add value labels
for bar, or_ in zip(bars, coef_df['Odds Ratio']):
    width = bar.get_width()
    ax2.text(width + 0.02, bar.get_y() + bar.get_height()/2,
            f'{or_:.3f}', ha='left', va='center', fontsize=11)

plt.tight_layout()
plt.show()

---

## 13.6 Practical Application: Marketing Campaign Response

Let's apply logistic regression to another critical business problem: predicting which customers will respond to a marketing campaign. This helps optimize marketing spend by targeting the right customers.

### The Problem

A company wants to run a direct mail campaign. Sending materials costs money, so they want to target customers most likely to respond. Can we predict who will respond based on customer characteristics?

In [ ]:
# Load marketing response data
marketing_data = pd.read_csv(f"{DATA_URL}/W13/marketing_response.csv")

print("Marketing Response Dataset")
print("=" * 50)
print(f"Total customers: {len(marketing_data):,}")
print(f"\nSample data:")
display(marketing_data.head())

print(f"\nResponse Rate: {marketing_data['Responded'].mean()*100:.1f}%")
print("(Most customers don't respond - class imbalance!)")

In [ ]:
# Explore feature relationships
print("Feature Statistics by Response:")
display(marketing_data.groupby('Responded').mean().round(2))

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

features = ['Age', 'Income', 'PreviousPurchases', 'EmailOpens']

for ax, feature in zip(axes.flatten(), features):
    for resp in [0, 1]:
        data = marketing_data[marketing_data['Responded'] == resp][feature]
        label = 'Responded' if resp == 1 else 'No Response'
        color = '#2ecc71' if resp == 1 else '#e74c3c'
        ax.hist(data, bins=20, alpha=0.6, label=label, color=color, density=True)
    ax.set_xlabel(feature)
    ax.set_ylabel('Density')
    ax.set_title(f'{feature} by Response')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Build marketing response model
marketing_features = ['Age', 'Income', 'PreviousPurchases', 'EmailOpens']
X_mkt = marketing_data[marketing_features]
y_mkt = marketing_data['Responded']

# Split data
X_mkt_train, X_mkt_test, y_mkt_train, y_mkt_test = train_test_split(
    X_mkt, y_mkt, test_size=0.2, random_state=42, stratify=y_mkt
)

# Train model
mkt_model = LogisticRegression(random_state=42, max_iter=1000)
mkt_model.fit(X_mkt_train, y_mkt_train)

# Predictions
y_mkt_pred = mkt_model.predict(X_mkt_test)
y_mkt_prob = mkt_model.predict_proba(X_mkt_test)[:, 1]

print("Marketing Response Model")
print("=" * 50)
print(f"\nModel Performance:")
print(f"  Accuracy: {accuracy_score(y_mkt_test, y_mkt_pred)*100:.1f}%")

print("\nClassification Report:")
print(classification_report(y_mkt_test, y_mkt_pred, target_names=['No Response', 'Responded']))

In [ ]:
# Interpret coefficients
print("Marketing Model - Feature Interpretation")
print("=" * 60)

mkt_coef_df = pd.DataFrame({
    'Feature': marketing_features,
    'Coefficient': mkt_model.coef_[0],
    'Odds Ratio': np.exp(mkt_model.coef_[0])
})
display(mkt_coef_df.round(4))

print("\nBusiness Insights:")
for _, row in mkt_coef_df.iterrows():
    feature = row['Feature']
    or_ = row['Odds Ratio']
    
    if or_ > 1:
        effect = f"increases response odds by {(or_-1)*100:.1f}%"
    else:
        effect = f"decreases response odds by {(1-or_)*100:.1f}%"
    
    print(f"  • {feature}: Each unit increase {effect}")

In [ ]:
# Calculate ROI of targeted marketing
print("Marketing Campaign ROI Analysis")
print("=" * 60)

# Campaign economics
cost_per_contact = 5       # $5 per direct mail piece
revenue_per_response = 100  # $100 average order from responders

n_test = len(y_mkt_test)
actual_responders = y_mkt_test.sum()

print(f"\nTest set size: {n_test} customers")
print(f"Actual responders: {actual_responders} ({actual_responders/n_test*100:.1f}%)")

# Scenario 1: Mass marketing (contact everyone)
mass_cost = n_test * cost_per_contact
mass_revenue = actual_responders * revenue_per_response
mass_profit = mass_revenue - mass_cost
mass_roi = (mass_profit / mass_cost) * 100

print(f"\n--- Scenario 1: Mass Marketing (Contact Everyone) ---")
print(f"Cost: ${mass_cost:,}")
print(f"Revenue: ${mass_revenue:,}")
print(f"Profit: ${mass_profit:,}")
print(f"ROI: {mass_roi:.1f}%")

# Scenario 2: Targeted marketing (top 20% by probability)
top_pct = 0.20
n_target = int(n_test * top_pct)

# Sort by probability and select top prospects
sorted_indices = np.argsort(y_mkt_prob)[::-1]
top_indices = sorted_indices[:n_target]

targeted_responders = y_mkt_test.values[top_indices].sum()
targeted_cost = n_target * cost_per_contact
targeted_revenue = targeted_responders * revenue_per_response
targeted_profit = targeted_revenue - targeted_cost
targeted_roi = (targeted_profit / targeted_cost) * 100

print(f"\n--- Scenario 2: Targeted Marketing (Top {int(top_pct*100)}%) ---")
print(f"Customers contacted: {n_target}")
print(f"Responders captured: {targeted_responders} ({targeted_responders/actual_responders*100:.1f}% of all responders)")
print(f"Cost: ${targeted_cost:,}")
print(f"Revenue: ${targeted_revenue:,}")
print(f"Profit: ${targeted_profit:,}")
print(f"ROI: {targeted_roi:.1f}%")

print(f"\n--- Improvement ---")
print(f"Cost reduction: ${mass_cost - targeted_cost:,} ({(1 - targeted_cost/mass_cost)*100:.0f}% less)")
print(f"Profit improvement: ${targeted_profit - mass_profit:,}")
print(f"ROI improvement: {targeted_roi - mass_roi:.1f} percentage points")

In [ ]:
# Visualize the lift from targeting
# Cumulative response rate at different targeting levels

# Sort all customers by predicted probability
sorted_idx = np.argsort(y_mkt_prob)[::-1]
sorted_actual = y_mkt_test.values[sorted_idx]

# Calculate cumulative metrics
cumulative_responses = np.cumsum(sorted_actual)
total_responses = sorted_actual.sum()
n_customers = len(sorted_actual)

pct_contacted = np.arange(1, n_customers + 1) / n_customers * 100
pct_responses_captured = cumulative_responses / total_responses * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Cumulative Response Curve (Lift Chart)
ax1 = axes[0]
ax1.plot(pct_contacted, pct_responses_captured, 'b-', linewidth=2, label='Model Targeting')
ax1.plot([0, 100], [0, 100], 'r--', linewidth=1.5, label='Random Selection')
ax1.fill_between(pct_contacted, pct_responses_captured, pct_contacted, alpha=0.2, color='blue')
ax1.set_xlabel('% of Customers Contacted', fontsize=12)
ax1.set_ylabel('% of Responders Captured', fontsize=12)
ax1.set_title('Cumulative Response Curve (Lift Chart)', fontsize=14)
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# Mark the 20% point
idx_20 = int(n_customers * 0.2)
ax1.plot(20, pct_responses_captured[idx_20], 'go', markersize=10)
ax1.annotate(f'Top 20% captures\n{pct_responses_captured[idx_20]:.0f}% of responders',
            xy=(20, pct_responses_captured[idx_20]), 
            xytext=(35, pct_responses_captured[idx_20]-15),
            fontsize=10, arrowprops=dict(arrowstyle='->', color='green'))

# Plot 2: Lift at different targeting levels
ax2 = axes[1]
lift = pct_responses_captured / pct_contacted
ax2.plot(pct_contacted, lift, 'g-', linewidth=2)
ax2.axhline(y=1, color='red', linestyle='--', label='Baseline (Lift = 1)')
ax2.set_xlabel('% of Customers Contacted', fontsize=12)
ax2.set_ylabel('Lift', fontsize=12)
ax2.set_title('Model Lift at Different Targeting Levels', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, max(lift) * 1.1)

plt.tight_layout()
plt.show()

print(f"\nAt top 20% targeting:")
print(f"  Lift = {lift[idx_20]:.2f}x (model captures {lift[idx_20]:.2f}x more responders than random)")

---

## 13.7 Practical Application: Loan Default Prediction

Our third application demonstrates logistic regression in the financial services industry: predicting loan defaults. This is critical for risk management and lending decisions.

In [ ]:
# Load loan default data
loan_data = pd.read_csv(f"{DATA_URL}/W13/loan_defaults.csv")

print("Loan Default Dataset")
print("=" * 50)
print(f"Total loans: {len(loan_data):,}")
print(f"Default rate: {loan_data['Defaulted'].mean()*100:.1f}%")
print(f"\nSample data:")
display(loan_data.head())
print(f"\nFeature statistics:")
display(loan_data.describe().round(2))

In [ ]:
# Build loan default model
loan_features = ['Income', 'DebtRatio', 'CreditScore', 'LoanAmount']
X_loan = loan_data[loan_features]
y_loan = loan_data['Defaulted']

# Split
X_loan_train, X_loan_test, y_loan_train, y_loan_test = train_test_split(
    X_loan, y_loan, test_size=0.2, random_state=42, stratify=y_loan
)

# Train
loan_model = LogisticRegression(random_state=42, max_iter=1000)
loan_model.fit(X_loan_train, y_loan_train)

# Predict
y_loan_pred = loan_model.predict(X_loan_test)
y_loan_prob = loan_model.predict_proba(X_loan_test)[:, 1]

print("Loan Default Model Performance")
print("=" * 50)
print(f"Accuracy: {accuracy_score(y_loan_test, y_loan_pred)*100:.1f}%")
print("\nClassification Report:")
print(classification_report(y_loan_test, y_loan_pred, target_names=['Repaid', 'Defaulted']))

In [ ]:
# Interpret loan model coefficients
print("Loan Default Risk Factors")
print("=" * 60)

loan_coef_df = pd.DataFrame({
    'Feature': loan_features,
    'Coefficient': loan_model.coef_[0],
    'Odds Ratio': np.exp(loan_model.coef_[0])
}).sort_values('Coefficient', key=abs, ascending=False)

display(loan_coef_df.round(4))

print("\nRisk Factor Interpretation:")
print("-" * 60)
print("Factors that INCREASE default risk (OR > 1):")
for _, row in loan_coef_df[loan_coef_df['Odds Ratio'] > 1].iterrows():
    print(f"  • {row['Feature']}: OR = {row['Odds Ratio']:.4f}")

print("\nFactors that DECREASE default risk (OR < 1):")
for _, row in loan_coef_df[loan_coef_df['Odds Ratio'] < 1].iterrows():
    print(f"  • {row['Feature']}: OR = {row['Odds Ratio']:.6f}")

In [ ]:
# Practical application: Loan approval decision support
print("Loan Application Risk Assessment")
print("=" * 60)

# Sample loan applications
applications = pd.DataFrame({
    'Applicant': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'Income': [75000, 45000, 95000, 55000],
    'DebtRatio': [0.25, 0.55, 0.30, 0.70],
    'CreditScore': [720, 620, 750, 580],
    'LoanAmount': [15000, 25000, 20000, 30000]
})

# Predict default probability
X_applications = applications[loan_features]
applications['Default_Prob'] = loan_model.predict_proba(X_applications)[:, 1]
applications['Risk_Level'] = applications['Default_Prob'].apply(
    lambda p: 'Low' if p < 0.2 else ('Medium' if p < 0.5 else 'High')
)
applications['Recommendation'] = applications['Default_Prob'].apply(
    lambda p: 'Approve' if p < 0.3 else ('Review' if p < 0.6 else 'Decline')
)

display(applications)

print("\nDecision Rules:")
print("  • Default Prob < 30%: Approve")
print("  • Default Prob 30-60%: Manual Review")
print("  • Default Prob > 60%: Decline")

---

## 13.8 Common Pitfalls and Best Practices

As you apply logistic regression in practice, be aware of these common issues and best practices:

### Pitfall 1: Ignoring Class Imbalance

When one class is much more common than the other (e.g., 95% non-churners, 5% churners), the model may simply predict the majority class for everyone.

**Solutions:**
- Use `stratify` parameter in train_test_split
- Consider class weights: `LogisticRegression(class_weight='balanced')`
- Use resampling techniques (oversampling minority, undersampling majority)

### Pitfall 2: Using Accuracy Alone

As discussed, accuracy can be misleading. Always examine the confusion matrix, precision, recall, and F1-score.

### Pitfall 3: Not Scaling Features

While logistic regression doesn't strictly require feature scaling, it can help with:
- Faster convergence
- Easier coefficient interpretation
- Regularization fairness

### Pitfall 4: Multicollinearity

Highly correlated features can make coefficients unstable and hard to interpret. Check correlations before modeling.

### Pitfall 5: Overfitting

With many features, logistic regression can overfit. Consider:
- Regularization: `LogisticRegression(penalty='l2', C=0.1)`
- Feature selection
- Cross-validation

### Best Practices Summary

| Practice | Why It Matters |
|----------|---------------|
| Explore data first | Understand distributions and relationships |
| Check class balance | Avoid misleading accuracy |
| Use stratified splits | Maintain class proportions |
| Examine multiple metrics | Get complete performance picture |
| Interpret coefficients | Gain business insights |


In [ ]:
# Demonstrating class_weight='balanced' for imbalanced data
print("Handling Class Imbalance")
print("=" * 60)

# Create a more imbalanced scenario (keep only 10% of churners)
np.random.seed(42)
churn_idx = churn_data[churn_data['Churned'] == 1].index
keep_churn = np.random.choice(churn_idx, size=len(churn_idx)//5, replace=False)
imbalanced_data = pd.concat([
    churn_data[churn_data['Churned'] == 0],
    churn_data.loc[keep_churn]
])

print(f"Imbalanced dataset:")
print(f"  Class 0 (Stay): {(imbalanced_data['Churned']==0).sum()}")
print(f"  Class 1 (Churn): {(imbalanced_data['Churned']==1).sum()}")
print(f"  Churn rate: {imbalanced_data['Churned'].mean()*100:.1f}%")

# Split imbalanced data
X_imb = imbalanced_data[feature_columns]
y_imb = imbalanced_data['Churned']
X_imb_train, X_imb_test, y_imb_train, y_imb_test = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42, stratify=y_imb
)

# Model without class weights
model_unweighted = LogisticRegression(random_state=42, max_iter=1000)
model_unweighted.fit(X_imb_train, y_imb_train)
y_pred_unweighted = model_unweighted.predict(X_imb_test)

# Model with balanced class weights
model_balanced = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
model_balanced.fit(X_imb_train, y_imb_train)
y_pred_balanced = model_balanced.predict(X_imb_test)

print("\n--- Unweighted Model ---")
print(f"Accuracy: {accuracy_score(y_imb_test, y_pred_unweighted)*100:.1f}%")
print(f"Recall (Churn): {recall_score(y_imb_test, y_pred_unweighted)*100:.1f}%")
cm_unweighted = confusion_matrix(y_imb_test, y_pred_unweighted)
print(f"Churners caught: {cm_unweighted[1,1]} / {cm_unweighted[1,:].sum()}")

print("\n--- Balanced Class Weight Model ---")
print(f"Accuracy: {accuracy_score(y_imb_test, y_pred_balanced)*100:.1f}%")
print(f"Recall (Churn): {recall_score(y_imb_test, y_pred_balanced)*100:.1f}%")
cm_balanced = confusion_matrix(y_imb_test, y_pred_balanced)
print(f"Churners caught: {cm_balanced[1,1]} / {cm_balanced[1,:].sum()}")

print("\n→ The balanced model catches more churners, even if accuracy is slightly lower!")

---

## 13.9 Ethical Implications of Classification Models

As we build models that make yes/no decisions about people—who gets a loan, who receives a retention offer, who is flagged as a fraud risk—we must grapple with profound ethical questions. Classification models can have life-altering consequences, and as the builder of these systems, you bear responsibility for their impacts.

### Why Classification Ethics Matter More Than Ever

Classification models are increasingly used for high-stakes decisions:

| Application | What's Being Decided | Who's Affected |
|-------------|---------------------|----------------|
| Loan approval | Access to credit | Aspiring homeowners, entrepreneurs |
| Hiring screening | Job opportunities | Job seekers, their families |
| Insurance pricing | Access and affordability | Everyone needing coverage |
| Fraud detection | Account freezes, investigations | Legitimate customers flagged wrongly |
| Healthcare triage | Treatment priority | Patients |
| Recidivism prediction | Criminal sentencing, parole | Defendants, communities |

**The common thread**: These models affect people's lives, and errors are not abstract statistics—they're denied opportunities, damaged finances, and altered life trajectories.

### The Asymmetry of Classification Errors

In our confusion matrix, we talked about false positives (FP) and false negatives (FN) as technical metrics. But consider their human meaning:

**For a loan default model:**
- **False Positive (predicting default when customer would repay)**: Denying credit to a creditworthy person. They can't buy a home, start a business, or access opportunity.
- **False Negative (predicting repayment when customer defaults)**: Approving a loan that defaults. The bank loses money.

Which error is worse? Financially, the bank cares more about false negatives. But ethically? Denying opportunity to deserving people—especially if this happens disproportionately to certain groups—raises serious fairness concerns.

### Disparate Impact: When Fair Models Aren't Fair

A model can be **technically accurate** yet **ethically problematic** if it produces different error rates for different groups.

**Example: A Hiring Model**

Suppose your model predicts "successful employee" using factors like:
- University attended
- Years of experience
- Previous job titles
- Zip code (residential area)

This model might be highly accurate overall but:
- Systematically disadvantage candidates from lower-income backgrounds
- Penalize career gaps (affecting parents, caregivers, those with health issues)
- Favor candidates from universities with historically limited access to certain groups

**Legal standard**: In many jurisdictions, if a model's error rate differs significantly between protected groups, it may constitute illegal discrimination—even if the model never explicitly uses race, gender, or other protected characteristics.

### Transparency and Explainability

One advantage of logistic regression over more complex models is **explainability**. You can tell someone:

*"Your loan application was declined because the model found your debt-to-income ratio and credit score combination indicates a 62% probability of default, which our risk policy considers too high to approve."*

This transparency enables:
- **Appeal**: The person can dispute incorrect data or provide context
- **Improvement**: They know what to work on
- **Accountability**: Decisions can be reviewed and challenged

Many jurisdictions now require explanations for automated decisions (GDPR's "right to explanation," US fair lending laws, etc.).

### Feedback Loops: When Models Create Their Own Reality

Classification models can create self-fulfilling prophecies:

**Example: Predictive Policing**
1. Model predicts more crime in certain neighborhoods (based on historical arrests)
2. Police patrol those neighborhoods more heavily
3. More arrests occur in those neighborhoods (because more police are there)
4. Model is "validated" and reinforced
5. Cycle continues, regardless of actual crime distribution

**Example: Credit Scoring**
1. Model denies credit to certain zip codes
2. Residents can't access credit to build businesses, buy homes
3. Economic conditions in those areas deteriorate
4. Model's predictions are "confirmed"

As a model builder, you must ask: **Is my model predicting reality or creating it?**

### Practical Steps for Ethical Classification

1. **Audit for Disparate Impact**
   - Calculate error rates (FPR, FNR) for different demographic groups
   - If error rates differ substantially, investigate and address

2. **Consider the Human Impact of Errors**
   - Who is harmed by false positives?
   - Who is harmed by false negatives?
   - Are certain groups disproportionately affected?

3. **Ensure Transparency**
   - Can you explain individual decisions?
   - Can affected people understand and challenge decisions?
   - Are the factors used legally and ethically appropriate?

4. **Test for Proxy Discrimination**
   - Are seemingly neutral features actually proxying for protected characteristics?
   - Would your model perform the same if you removed potentially discriminatory features?

5. **Build in Human Oversight**
   - Should there be human review for edge cases?
   - Who is accountable for model decisions?
   - How can errors be appealed and corrected?

6. **Monitor Deployed Models**
   - Track outcomes across demographic groups over time
   - Watch for feedback loops and unintended consequences
   - Retrain with awareness of these issues

### The Broader Context: AI and Social Responsibility

You are entering a profession at a pivotal moment. AI and machine learning systems are being deployed across society, making decisions that were once made by humans. With this power comes responsibility:

- **Technical excellence is not enough.** A model can be statistically sophisticated and ethically catastrophic.
- **Intent doesn't determine impact.** You may not intend to discriminate, but your model might.
- **Objectivity is a myth.** Every model reflects choices about what to measure, how to weight outcomes, and whose interests matter.

The analysts and data scientists who will make the biggest positive impact are those who combine technical skill with ethical awareness—who ask not just "Can we build this?" but "Should we build this? For whom? With what safeguards?"

### Questions Every Classification Model Should Answer

Before deploying any classification model that affects people's lives, you should be able to answer:

1. What decision does this model enable?
2. Who is affected by this decision?
3. What happens to people who receive a positive classification? Negative?
4. What are the consequences of each type of error?
5. Do error rates differ across demographic groups?
6. Can affected people understand and challenge decisions?
7. Who is accountable if the model causes harm?
8. How will you monitor for unintended consequences?

If you cannot answer these questions, you are not ready to deploy the model.

---

## 13.10 Complete Workflow: A Business Case Study

Let's bring everything together with a complete, end-to-end workflow for a realistic business scenario.

### Scenario: Customer Retention at TeleCo

TeleCo is a telecommunications company experiencing high customer churn. They want to:
1. Identify customers at risk of churning
2. Understand what drives churn
3. Target at-risk customers with retention offers
4. Measure the business impact

In [ ]:
# COMPLETE WORKFLOW: Customer Churn Prediction at TeleCo

print("="*70)
print("TeleCo Customer Churn Analysis")
print("Complete Machine Learning Workflow")
print("="*70)

# ============================================
# STEP 1: Load and Explore Data
# ============================================
print("\n📊 STEP 1: Data Loading and Exploration")
print("-"*50)

# Load data
teleco_data = pd.read_csv(f"{DATA_URL}/W13/customer_churn.csv")
print(f"Dataset loaded: {len(teleco_data):,} customers")
print(f"Churn rate: {teleco_data['Churned'].mean()*100:.1f}%")
print(f"\nFeatures: {list(teleco_data.columns)}")

In [ ]:
# ============================================
# STEP 2: Data Preparation
# ============================================
print("\n🔧 STEP 2: Data Preparation")
print("-"*50)

# Define features and target
feature_cols = ['Tenure', 'MonthlyCharges', 'SupportCalls']
X = teleco_data[feature_cols]
y = teleco_data['Churned']

# Split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)} samples (churn rate: {y_train.mean()*100:.1f}%)")
print(f"Test set: {len(X_test)} samples (churn rate: {y_test.mean()*100:.1f}%)")

In [ ]:
# ============================================
# STEP 3: Model Training
# ============================================
print("\n🤖 STEP 3: Model Training")
print("-"*50)

# Create and train model
churn_model = LogisticRegression(random_state=42, max_iter=1000)
churn_model.fit(X_train, y_train)

print("LogisticRegression model trained successfully!")
print(f"\nModel equation:")
print(f"  log(odds) = {churn_model.intercept_[0]:.3f}", end='')
for feat, coef in zip(feature_cols, churn_model.coef_[0]):
    sign = '+' if coef >= 0 else ''
    print(f" {sign}{coef:.4f}*{feat}", end='')
print()

In [ ]:
# ============================================
# STEP 4: Model Evaluation
# ============================================
print("\n📈 STEP 4: Model Evaluation")
print("-"*50)

# Predictions
y_pred = churn_model.predict(X_test)
y_prob = churn_model.predict_proba(X_test)[:, 1]

# Metrics
print(f"\nPerformance Metrics:")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred)*100:.1f}%")
print(f"  Precision: {precision_score(y_test, y_pred)*100:.1f}%")
print(f"  Recall:    {recall_score(y_test, y_pred)*100:.1f}%")
print(f"  F1-Score:  {f1_score(y_test, y_pred)*100:.1f}%")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f"\nConfusion Matrix:")
print(f"  True Negatives:  {tn} (correctly predicted Stay)")
print(f"  False Positives: {fp} (incorrectly predicted Churn)")
print(f"  False Negatives: {fn} (missed Churners - the costly ones!)")
print(f"  True Positives:  {tp} (correctly predicted Churn)")

In [ ]:
# ============================================
# STEP 5: Feature Interpretation
# ============================================
print("\n🔍 STEP 5: Feature Interpretation")
print("-"*50)

interpretation_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': churn_model.coef_[0],
    'Odds_Ratio': np.exp(churn_model.coef_[0])
})

print("\nChurn Risk Factors:")
for _, row in interpretation_df.iterrows():
    or_ = row['Odds_Ratio']
    if or_ > 1:
        pct = (or_ - 1) * 100
        print(f"  📈 {row['Feature']}: Increases churn odds by {pct:.1f}% per unit")
    else:
        pct = (1 - or_) * 100
        print(f"  📉 {row['Feature']}: Decreases churn odds by {pct:.1f}% per unit")

In [ ]:
# ============================================
# STEP 6: Business Application
# ============================================
print("\n💼 STEP 6: Business Application")
print("-"*50)

# Identify high-risk customers
test_results = X_test.copy()
test_results['Actual_Churn'] = y_test.values
test_results['Churn_Probability'] = y_prob
test_results['Risk_Level'] = pd.cut(
    y_prob, 
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low', 'Medium', 'High']
)

# Risk distribution
print("\nCustomer Risk Distribution:")
risk_summary = test_results.groupby('Risk_Level').agg({
    'Tenure': 'count',
    'Actual_Churn': 'mean'
}).rename(columns={'Tenure': 'Count', 'Actual_Churn': 'Actual_Churn_Rate'})
risk_summary['Actual_Churn_Rate'] = (risk_summary['Actual_Churn_Rate'] * 100).round(1).astype(str) + '%'
display(risk_summary)

# High-risk customers for intervention
high_risk = test_results[test_results['Risk_Level'] == 'High']
print(f"\n⚠️ High-Risk Customers: {len(high_risk)}")
print(f"   These customers should receive retention offers!")

In [ ]:
# ============================================
# STEP 7: ROI Analysis
# ============================================
print("\n💰 STEP 7: ROI Analysis")
print("-"*50)

# Business assumptions
retention_offer_cost = 50       # Cost of retention offer per customer
avg_customer_value = 1000       # Annual value of a retained customer
retention_success_rate = 0.4   # 40% of at-risk customers stay if offered retention

# Without model (no intervention)
baseline_churners = y_test.sum()
baseline_loss = baseline_churners * avg_customer_value

# With model (intervene on high-risk)
n_interventions = len(high_risk)
intervention_cost = n_interventions * retention_offer_cost

# Of high-risk customers who would actually churn, some are retained
true_churners_in_high_risk = high_risk['Actual_Churn'].sum()
customers_retained = int(true_churners_in_high_risk * retention_success_rate)
retention_value = customers_retained * avg_customer_value

# Net benefit
net_benefit = retention_value - intervention_cost

print(f"\nWithout Model (No Intervention):")
print(f"  Expected churners: {baseline_churners}")
print(f"  Revenue loss: ${baseline_loss:,.0f}")

print(f"\nWith Model (Target High-Risk):")
print(f"  Customers targeted: {n_interventions}")
print(f"  Intervention cost: ${intervention_cost:,.0f}")
print(f"  True churners targeted: {true_churners_in_high_risk}")
print(f"  Customers retained: {customers_retained}")
print(f"  Retention value: ${retention_value:,.0f}")

print(f"\n📊 Net Benefit: ${net_benefit:,.0f}")
print(f"   ROI: {(net_benefit/intervention_cost)*100:.0f}%")

In [ ]:
# ============================================
# STEP 8: Recommendations
# ============================================
print("\n📋 STEP 8: Business Recommendations")
print("="*70)

print("""
Based on our analysis, we recommend:

1. IMMEDIATE ACTIONS:
   • Target high-risk customers (churn probability > 60%) with retention offers
   • Prioritize customers with high support call frequency
   • Focus retention efforts on newer customers (low tenure)

2. PROCESS IMPROVEMENTS:
   • Implement proactive support for customers with 3+ support calls
   • Create loyalty programs to increase customer tenure
   • Review pricing strategy for high-charge customers

3. MODEL DEPLOYMENT:
   • Score all customers monthly to identify at-risk segments
   • Automate retention offer triggers based on risk level
   • Monitor model performance and retrain quarterly

4. EXPECTED IMPACT:
   • Reduce churn by targeting the right customers
   • Improve retention ROI through smart targeting
   • Build data-driven customer relationship management
""")

print("="*70)
print("Analysis Complete!")
print("="*70)

---

## 13.11 Summary

### Key Concepts Covered

| Concept | Description |
|---------|-------------|
| **Classification** | Predicting categorical outcomes (vs. continuous for regression) |
| **Logistic Regression** | Algorithm that uses sigmoid function for binary classification |
| **Sigmoid Function** | Transforms linear combination to probability [0, 1] |
| **Odds Ratio** | Interpretable measure of feature effect on outcome |
| **Confusion Matrix** | TP, TN, FP, FN breakdown of predictions |
| **Precision** | Of predicted positives, how many are correct? |
| **Recall** | Of actual positives, how many did we find? |
| **F1-Score** | Harmonic mean of precision and recall |

### scikit-learn Workflow

```python
# 1. Import
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 2. Prepare data
X = df[['feature1', 'feature2', 'feature3']]
y = df['target']

# 3. Split data (with stratification for imbalanced classes)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Train model
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

# 5. Predict
y_pred = model.predict(X_test)           # Class labels
y_prob = model.predict_proba(X_test)[:, 1]  # Probabilities

# 6. Evaluate
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
```

### Business Applications

| Application | Target Variable | Key Features |
|-------------|-----------------|-------------|
| Customer Churn | Churned (0/1) | Tenure, charges, support calls |
| Marketing Response | Responded (0/1) | Age, income, purchase history |
| Loan Default | Defaulted (0/1) | Income, debt ratio, credit score |
| Fraud Detection | Fraudulent (0/1) | Transaction amount, location, time |
| Email Classification | Spam (0/1) | Word frequency, sender, links |

---

## Practice Exercises

Test your understanding with these exercises:

### Exercise 1: Build a Churn Model
Using the customer churn dataset, build a logistic regression model that:
- Uses all three features (Tenure, MonthlyCharges, SupportCalls)
- Evaluates performance with accuracy, precision, recall, and F1-score
- Interprets all coefficients as odds ratios in business terms

### Exercise 2: Marketing ROI
Using the marketing response dataset:
- Build a response prediction model
- Calculate the lift from targeting the top 30% vs random 30%
- Estimate the ROI improvement from model-based targeting

### Exercise 3: Loan Risk Assessment
Using the loan default dataset:
- Build a default prediction model
- Create risk categories (Low, Medium, High) based on default probability
- For a new loan application with Income=$60,000, DebtRatio=0.4, CreditScore=680, LoanAmount=$20,000:
  - Calculate the default probability
  - Make a recommendation (Approve/Review/Decline)

---

## Course Conclusion

Congratulations! You've completed the core machine learning content of ISM4641 Python for Business Analytics. Over these weeks, you've learned:

1. **Python Fundamentals** - Variables, data types, operators
2. **Control Flow** - Conditionals and loops
3. **Functions** - Modular, reusable code
4. **NumPy** - Numerical computing with arrays
5. **Pandas** - Data manipulation and analysis
6. **Matplotlib** - Data visualization
7. **Statistics** - Descriptive and inferential statistics
8. **Linear Regression** - Predicting continuous outcomes
9. **Logistic Regression** - Classifying categorical outcomes

These skills form the foundation of data science and business analytics. You're now equipped to:
- Load, clean, and explore data
- Build predictive models
- Evaluate model performance
- Communicate insights to stakeholders
- Make data-driven business decisions

**Your Next Steps:**
- Apply these skills to your final project
- Continue learning (more ML algorithms, deep learning, cloud deployment)
- Practice with real-world datasets
- Build a portfolio of data science projects

Best of luck in your future data science journey!

---
*ISM4641 Python for Business Analytics - Week 13: Logistic Regression*